# MLP Dry Bean — Experimentos (Q1–Q7)

Este notebook implementa o protocolo definido em `PROJECT_PLAN.md`.

**Split:** 70% desenvolvimento / 30% teste. O desenvolvimento é dividido em 80% treino / 20% validação (efetivo 56/14/30).
**Q1–Q5:** treino/validação. **Q6:** treino no desenvolvimento completo e avaliação no teste. **Q7:** 5-fold no desenvolvimento.
**Critério de convergência:** entropia cruzada de treino ≤ 0,001, com máximo de 400 épocas.
**Métrica principal:** macro-F1.

Execute as células em ordem. Para refazer uma questão, apague o CSV correspondente ou ligue a flag em `FORCE_RERUN`.

## 0. Configuração e imports

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, time, math, random, warnings, gc
from datetime import datetime

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Dry_Bean_Dataset.csv"
RESULTS_CSV = PROJECT_ROOT / "results" / "csv"
RESULTS_FIG = PROJECT_ROOT / "results" / "figures"
RESULTS_TAB = PROJECT_ROOT / "results" / "tables"
for p in (RESULTS_CSV, RESULTS_FIG, RESULTS_TAB):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_SHUFFLE_SEED = 123
Q1_EPOCHS = 100
MAX_EPOCHS = 400
CONVERGENCE_THRESHOLD = 1e-3
BATCH_SIZE = 32
N_CLASSES = 7
TARGET_COL = "Class"

FORCE_RERUN = {
    "q1": False, "q2": False, "q3": False, "q4": False,
    "q5": False, "q6": False, "q7": False,
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
PALETTE = sns.color_palette("colorblind")

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Seeds e determinismo

In [ ]:
def set_global_seed(seed=SEED):
    """Fixa as seeds de Python, NumPy e TensorFlow."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def enable_determinism():
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as exc:
        print("Aviso: não foi possível habilitar determinismo total:", exc)

def clear_session():
    tf.keras.backend.clear_session()
    gc.collect()

set_global_seed(SEED)
enable_determinism()
print("Seeds fixadas em", SEED)

## 2. Carregamento e validação dos dados

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("Nulos:", int(df.isna().sum().sum()))
print("Duplicatas:", int(df.duplicated().sum()))

assert TARGET_COL in df.columns
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]
assert len(FEATURE_COLS) == 16, FEATURE_COLS
assert len(df) == 13611, len(df)
assert df[TARGET_COL].nunique() == 7

class_counts = (df[TARGET_COL].value_counts().rename_axis("class").reset_index(name="count"))
class_counts.to_csv(RESULTS_CSV / "class_distribution.csv", index=False)
display(class_counts)

label_encoder = LabelEncoder()
label_encoder.fit(df[TARGET_COL].values)
class_names = list(label_encoder.classes_)
pd.DataFrame({"class_index": range(len(class_names)), "class_name": class_names}).to_csv(
    RESULTS_CSV / "class_mapping.csv", index=False)
print("Classes:", class_names)

# EDA mínima
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=class_counts, x="class", y="count", palette="colorblind", ax=ax)
ax.set_title("Distribuição das classes")
ax.set_xlabel("Classe")
ax.set_ylabel("Contagem")
ax.tick_params(axis="x", rotation=30)
fig.savefig(RESULTS_FIG / "eda_class_distribution.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[FEATURE_COLS].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Correlação de Pearson entre atributos")
fig.savefig(RESULTS_FIG / "eda_correlation.png", bbox_inches="tight")
plt.show()

## 3. Split 70/30 com validação interna 80/20

In [ ]:
X = df[FEATURE_COLS].astype("float32").values
y_int = label_encoder.transform(df[TARGET_COL].values)
y = tf.keras.utils.to_categorical(y_int, num_classes=N_CLASSES).astype("float32")
row_index = np.arange(len(df))

# 70% desenvolvimento / 30% teste
X_dev, X_test, y_dev, y_test, y_int_dev, y_int_test, idx_dev, idx_test = train_test_split(
    X, y, y_int, row_index,
    test_size=0.30, stratify=y_int, random_state=SEED)

# 80/20 do desenvolvimento -> 56% treino / 14% validação
X_train, X_val, y_train, y_val, y_int_train, y_int_val, idx_train, idx_val = train_test_split(
    X_dev, y_dev, y_int_dev, idx_dev,
    test_size=0.20, stratify=y_int_dev, random_state=SEED)

print(f"Treino: {X_train.shape[0]} | Validação: {X_val.shape[0]} | Teste: {X_test.shape[0]}")

split_df = pd.DataFrame({"row_index": row_index, "split": "?"})
split_df.loc[idx_train, "split"] = "train"
split_df.loc[idx_val, "split"] = "val"
split_df.loc[idx_test, "split"] = "test"
split_df["class"] = df[TARGET_COL].values
split_df.to_csv(RESULTS_CSV / "split_indices.csv", index=False)

# Scaler para Q1-Q5: ajustado somente no treino de 56%
scaler_train = StandardScaler()
X_train_scaled = scaler_train.fit_transform(X_train).astype("float32")
X_val_scaled = scaler_train.transform(X_val).astype("float32")
X_dev_scaled = scaler_train.transform(X_dev).astype("float32")
X_test_scaled = scaler_train.transform(X_test).astype("float32")

# Scaler para Q6: ajustado somente no desenvolvimento de 70%
scaler_dev = StandardScaler()
X_dev_scaled70 = scaler_dev.fit_transform(X_dev).astype("float32")
X_test_scaled70 = scaler_dev.transform(X_test).astype("float32")

print("Scalers ajustados. O scaler de Q1-Q5 usa 56% de treino; o de Q6 usa 70% de desenvolvimento.")

## 4. Funções reutilizáveis

In [ ]:
def make_dataset(X, y, training=False, seed=DATA_SHUFFLE_SEED):
    """Cria tf.data.Dataset com ordem determinística."""
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def build_mlp(input_dim, hidden_layers, learning_rate, momentum, seed=SEED):
    """Constrói uma MLP com ReLU nas ocultas e Softmax na saída."""
    set_global_seed(seed)
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(keras.layers.Dense(
            int(units), activation="relu",
            kernel_initializer="glorot_uniform",
            bias_initializer="zeros"))
    model.add(keras.layers.Dense(N_CLASSES, activation="softmax"))
    optimizer = keras.optimizers.SGD(
        learning_rate=float(learning_rate), momentum=float(momentum), nesterov=False)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model


class ConvergenceCallback(keras.callbacks.Callback):
    """Para o treinamento quando a loss de treino atinge o limiar."""
    def __init__(self, threshold=CONVERGENCE_THRESHOLD):
        super().__init__()
        self.threshold = threshold
        self.converged = False
        self.converged_epoch = None
        self.diverged = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get("loss", np.nan)
        if not np.isfinite(loss):
            self.diverged = True
            self.model.stop_training = True
            return
        if loss <= self.threshold:
            self.converged = True
            self.converged_epoch = epoch + 1
            self.model.stop_training = True


class TestMetricsCallback(keras.callbacks.Callback):
    """Registra métricas de teste por época apenas para visualização."""
    def __init__(self, X_test, y_test):
        super().__init__()
        self.X_test = X_test
        self.y_test = y_test
        self.test_loss = []
        self.test_accuracy = []

    def on_epoch_end(self, epoch, logs=None):
        res = self.model.evaluate(self.X_test, self.y_test, verbose=0)
        self.test_loss.append(float(res[0]))
        self.test_accuracy.append(float(res[1]))


def predict_labels(model, X, y):
    probs = model.predict(X, verbose=0)
    return np.argmax(y, axis=1), np.argmax(probs, axis=1)


def evaluate_model(model, X, y):
    loss, acc = model.evaluate(X, y, verbose=0)
    y_true, y_pred = predict_labels(model, X, y)
    return {
        "loss": float(loss),
        "accuracy": float(acc),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def history_to_df(history, **meta):
    dfh = pd.DataFrame(history.history).reset_index().rename(columns={"index": "epoch"})
    dfh["epoch"] = dfh["epoch"] + 1
    for k, v in meta.items():
        dfh[k] = v
    return dfh


def select_by_rule(rule, X, y_int, feature_names):
    """Seleciona atributos por uma regra reproduzível, usando apenas os dados recebidos."""
    n = len(feature_names)
    if rule == "all_16":
        return list(feature_names)

    mi = mutual_info_classif(X, y_int, random_state=SEED)
    var = np.var(X, axis=0)
    corr = np.corrcoef(X, rowvar=False) if n > 1 else np.array([[1.0]])

    def low_redundancy():
        parent = list(range(n))
        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
        for i in range(n):
            for j in range(i + 1, n):
                if abs(corr[i, j]) >= 0.95:
                    union(i, j)
        groups = {}
        for i in range(n):
            groups.setdefault(find(i), []).append(i)
        keep = []
        for group in groups.values():
            best = sorted(group, key=lambda i: (-mi[i], feature_names[i]))[0]
            keep.append(best)
        return sorted(keep)

    if rule == "low_redundancy":
        idx = low_redundancy()
    elif rule == "top_10_mi":
        idx = sorted(range(n), key=lambda i: (-mi[i], feature_names[i]))[:10]
    elif rule == "compact":
        survivors = low_redundancy()
        survivors = sorted(survivors, key=lambda i: (-mi[i], feature_names[i]))[:10]
        idx = sorted(survivors)
    elif rule == "no_noise":
        threshold_var = np.percentile(var, 10)
        threshold_mi = np.median(mi)
        noisy = [i for i in range(n) if var[i] <= threshold_var and mi[i] <= threshold_mi]
        idx = [i for i in range(n) if i not in noisy]
        if len(idx) == n:
            idx = list(range(n))
    else:
        raise ValueError(f"Regra desconhecida: {rule}")
    return [feature_names[i] for i in sorted(idx)]


def feature_indices(features, feature_names):
    return [feature_names.index(f) for f in features]


def savefig(fig, name):
    fig.savefig(RESULTS_FIG / name, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 5. Q1 — Exploração inicial

Topologia 16-60-7; ReLU/Softmax; SGD lr=0,01 e momento=0,9; 100 épocas; cinco seeds; somente o treino de 56%.

In [ ]:
Q1_SEEDS = [42, 43, 44, 45, 46]
Q1_HIDDEN = [60]
Q1_LR, Q1_MOM = 0.01, 0.9
q1_runs_path = RESULTS_CSV / "q1_runs.csv"
q1_hist_path = RESULTS_CSV / "q1_history.csv"

if q1_runs_path.exists() and q1_hist_path.exists() and not FORCE_RERUN["q1"]:
    q1_runs = pd.read_csv(q1_runs_path)
    q1_hist = pd.read_csv(q1_hist_path)
    print("Q1 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    for seed in Q1_SEEDS:
        clear_session()
        model = build_mlp(X_train_scaled.shape[1], Q1_HIDDEN, Q1_LR, Q1_MOM, seed=seed)
        train_ds = make_dataset(X_train_scaled, y_train, training=True)
        t0 = time.perf_counter()
        hist = model.fit(train_ds, epochs=Q1_EPOCHS, verbose=0)
        elapsed = time.perf_counter() - t0
        metrics = evaluate_model(model, X_train_scaled, y_train)
        losses = np.asarray(hist.history["loss"])
        run_id = f"q1_s{seed}"
        run_rows.append({
            "run_id": run_id, "seed": seed, "epochs": len(losses),
            "loss_final": float(losses[-1]), "accuracy_final": metrics["accuracy"],
            "loss_min": float(np.min(losses)), "loss_min_epoch": int(np.argmin(losses) + 1),
            "macro_f1_final": metrics["macro_f1"], "time_s": elapsed,
        })
        h = history_to_df(hist, run_id=run_id, seed=seed)
        hist_rows.append(h)
        print(f"{run_id}: loss={losses[-1]:.4f} acc={metrics['accuracy']:.4f} tempo={elapsed:.1f}s")
    q1_runs = pd.DataFrame(run_rows)
    q1_hist = pd.concat(hist_rows, ignore_index=True)
    q1_runs.to_csv(q1_runs_path, index=False)
    q1_hist.to_csv(q1_hist_path, index=False)

display(q1_runs.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
for run_id, g in q1_hist.groupby("run_id"):
    ax.plot(g["epoch"], g["loss"], label=run_id, alpha=0.8)
ax.set_title("Q1 — Loss de treino por inicialização")
ax.set_xlabel("Época"); ax.set_ylabel("Categorical cross-entropy")
ax.legend(ncol=2, fontsize=8)
savefig(fig, "q1_loss_curves.png")

fig, ax = plt.subplots(figsize=(9, 5))
for run_id, g in q1_hist.groupby("run_id"):
    ax.plot(g["epoch"], g["accuracy"], label=run_id, alpha=0.8)
ax.set_title("Q1 — Acurácia de treino por inicialização")
ax.set_xlabel("Época"); ax.set_ylabel("Acurácia")
ax.legend(ncol=2, fontsize=8)
savefig(fig, "q1_accuracy_curves.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=q1_runs, x="seed", y="loss_final", ax=axes[0], palette="colorblind")
axes[0].set_title("Loss final por seed")
sns.barplot(data=q1_runs, x="seed", y="accuracy_final", ax=axes[1], palette="colorblind")
axes[1].set_title("Acurácia final por seed")
axes[1].set_ylim(0, 1.05)
savefig(fig, "q1_final_metrics.png")

**Análise Q1.** As curvas permitem comparar velocidade de convergência e dispersão entre inicializações. A variabilidade das métricas finais indica a sensibilidade ao ponto de partida; como apenas o treino é usado, não é possível confirmar overfitting, apenas discutir sinais nas curvas. A topologia 16-60-7 e as ativações ReLU/Softmax servem de base para a Q2.

## 6. Q2 — Learning rate e momento

Grid 3 × 3: lr ∈ {0,01; 0,1; 0,5} e momento ∈ {0,5; 0,7; 0,9}. Máximo de 400 épocas; parada quando a loss de treino ≤ 0,001.

In [ ]:
Q2_LRS = [0.01, 0.1, 0.5]
Q2_MOMENTA = [0.5, 0.7, 0.9]
q2_grid_path = RESULTS_CSV / "q2_grid.csv"
q2_hist_path = RESULTS_CSV / "q2_history.csv"

if q2_grid_path.exists() and q2_hist_path.exists() and not FORCE_RERUN["q2"]:
    q2_grid = pd.read_csv(q2_grid_path)
    q2_hist = pd.read_csv(q2_hist_path)
    print("Q2 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    for lr in Q2_LRS:
        for mom in Q2_MOMENTA:
            run_id = f"q2_lr{lr}_m{mom}"
            clear_session()
            model = build_mlp(X_train_scaled.shape[1], Q1_HIDDEN, lr, mom, seed=SEED)
            cb = ConvergenceCallback()
            train_ds = make_dataset(X_train_scaled, y_train, training=True)
            t0 = time.perf_counter()
            hist = model.fit(train_ds, validation_data=(X_val_scaled, y_val),
                             epochs=MAX_EPOCHS, callbacks=[cb], verbose=0)
            elapsed = time.perf_counter() - t0
            n_epochs = len(hist.history["loss"])
            if cb.diverged or not np.isfinite(hist.history["loss"][-1]):
                status = "diverged"
                tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
                va_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
            else:
                status = "ok"
                tr_metrics = evaluate_model(model, X_train_scaled, y_train)
                va_metrics = evaluate_model(model, X_val_scaled, y_val)
            run_rows.append({
                "run_id": run_id, "learning_rate": lr, "momentum": mom,
                "status": status, "converged": bool(cb.converged),
                "epochs": n_epochs,
                "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
                "loss_val": va_metrics["loss"], "acc_val": va_metrics["accuracy"],
                "macro_f1_val": va_metrics["macro_f1"],
                "weighted_f1_val": va_metrics["weighted_f1"],
                "time_s": elapsed,
            })
            h = history_to_df(hist, run_id=run_id, learning_rate=lr, momentum=mom)
            hist_rows.append(h)
            print(f"{run_id}: status={status} converged={cb.converged} epochs={n_epochs} "
                  f"val_f1={va_metrics['macro_f1']:.4f}")
    q2_grid = pd.DataFrame(run_rows)
    q2_hist = pd.concat(hist_rows, ignore_index=True)
    q2_grid.to_csv(q2_grid_path, index=False)
    q2_hist.to_csv(q2_hist_path, index=False)

# Seleção
valid = q2_grid[(q2_grid["status"] == "ok") & np.isfinite(q2_grid["macro_f1_val"])].copy()
if len(valid) == 0:
    raise RuntimeError("Q2 sem configurações válidas.")
converged = valid[valid["converged"] == True]
pool = converged if len(converged) else valid
best_f1 = pool["macro_f1_val"].max()
candidates = pool[pool["macro_f1_val"] >= best_f1 - 0.01]
q2_winner = candidates.sort_values(["epochs", "loss_val"], ascending=[True, True]).iloc[0]
q2_grid["selected"] = q2_grid["run_id"].eq(q2_winner["run_id"])
q2_grid.to_csv(q2_grid_path, index=False)
q2_lr, q2_mom = float(q2_winner["learning_rate"]), float(q2_winner["momentum"])
print("Q2 vencedora:", q2_winner["run_id"], "| lr=", q2_lr, "momento=", q2_mom)
display(q2_grid.sort_values(["selected", "macro_f1_val"], ascending=[False, False]).round(4))

pivot_epochs = q2_grid.pivot(index="momentum", columns="learning_rate", values="epochs")
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot_epochs, annot=True, fmt=".0f", cmap="viridis", ax=ax)
ax.set_title("Q2 — Épocas até convergência")
ax.set_xlabel("Learning rate"); ax.set_ylabel("Momentum")
savefig(fig, "q2_heatmap_epochs.png")

pivot_f1 = q2_grid.pivot(index="momentum", columns="learning_rate", values="macro_f1_val")
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Q2 — Macro-F1 de validação")
ax.set_xlabel("Learning rate"); ax.set_ylabel("Momentum")
savefig(fig, "q2_heatmap_val_f1.png")

fig, ax = plt.subplots(figsize=(10, 6))
for run_id, g in q2_hist.groupby("run_id"):
    ax.plot(g["epoch"], g["loss"], label=run_id, alpha=0.65, linewidth=1)
ax.set_yscale("log")
ax.set_title("Q2 — Loss de treino por configuração")
ax.set_xlabel("Época"); ax.set_ylabel("Categorical cross-entropy (log)")
ax.legend(fontsize=6, ncol=2)
savefig(fig, "q2_loss_curves.png")

**Análise Q2.** Taxas altas tendem a divergir ou oscilar; taxas baixas tendem a convergir lentamente. O momento intermediário costuma equilibrar velocidade e estabilidade. A configuração vencedora é usada na Q3–Q7.

## 7. Q3 — Topologia

1 a 3 camadas ocultas; 10 a 100 neurônios em passos de 10, com largura uniforme; 30 configurações. Hiperparâmetros da Q2; treino/validação; máximo 400 épocas.

In [ ]:
Q3_LAYERS = [1, 2, 3]
Q3_NEURONS = list(range(10, 101, 10))
q3_grid_path = RESULTS_CSV / "q3_grid.csv"
q3_hist_path = RESULTS_CSV / "q3_history.csv"

if q3_grid_path.exists() and q3_hist_path.exists() and not FORCE_RERUN["q3"]:
    q3_grid = pd.read_csv(q3_grid_path)
    q3_hist = pd.read_csv(q3_hist_path)
    print("Q3 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    for n_layers in Q3_LAYERS:
        for neurons in Q3_NEURONS:
            run_id = f"q3_l{n_layers}_n{neurons}_s{SEED}"
            clear_session()
            model = build_mlp(X_train_scaled.shape[1], [neurons] * n_layers,
                              q2_lr, q2_mom, seed=SEED)
            cb = ConvergenceCallback()
            train_ds = make_dataset(X_train_scaled, y_train, training=True)
            t0 = time.perf_counter()
            hist = model.fit(train_ds, validation_data=(X_val_scaled, y_val),
                             epochs=MAX_EPOCHS, callbacks=[cb], verbose=0)
            elapsed = time.perf_counter() - t0
            n_epochs = len(hist.history["loss"])
            if cb.diverged or not np.isfinite(hist.history["loss"][-1]):
                status = "diverged"
                tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
                va_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
            else:
                status = "ok"
                tr_metrics = evaluate_model(model, X_train_scaled, y_train)
                va_metrics = evaluate_model(model, X_val_scaled, y_val)
            run_rows.append({
                "run_id": run_id, "n_layers": n_layers, "neurons": neurons,
                "n_params": int(model.count_params()), "status": status,
                "converged": bool(cb.converged), "epochs": n_epochs,
                "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
                "loss_val": va_metrics["loss"], "acc_val": va_metrics["accuracy"],
                "macro_f1_val": va_metrics["macro_f1"],
                "weighted_f1_val": va_metrics["weighted_f1"],
                "gap": va_metrics["loss"] - tr_metrics["loss"] if status == "ok" else np.nan,
                "time_s": elapsed,
            })
            h = history_to_df(hist, run_id=run_id, n_layers=n_layers, neurons=neurons)
            hist_rows.append(h)
            print(f"{run_id}: status={status} epochs={n_epochs} "
                  f"val_f1={va_metrics['macro_f1']:.4f} tempo={elapsed:.1f}s")
    q3_grid = pd.DataFrame(run_rows)
    q3_hist = pd.concat(hist_rows, ignore_index=True)
    q3_grid.to_csv(q3_grid_path, index=False)
    q3_hist.to_csv(q3_hist_path, index=False)

# Ranking
valid = q3_grid[(q3_grid["status"] == "ok") & np.isfinite(q3_grid["macro_f1_val"])].copy()
valid = valid.sort_values(["macro_f1_val", "loss_val", "time_s"], ascending=[False, True, True])
valid["rank"] = range(1, len(valid) + 1)
valid["selected_top4"] = valid["rank"] <= 4
q3_grid = q3_grid.drop(columns=[c for c in ["rank", "selected_top4"] if c in q3_grid.columns]).merge(
    valid[["run_id", "rank", "selected_top4"]], on="run_id", how="left")
q3_grid.to_csv(q3_grid_path, index=False)
q3_top4 = q3_grid[q3_grid["selected_top4"] == True].sort_values("rank")
print("Top 4 da Q3:")
display(q3_top4[["rank", "run_id", "n_layers", "neurons", "macro_f1_val", "loss_val", "time_s"]].round(4))

pivot_f1 = q3_grid.pivot(index="n_layers", columns="neurons", values="macro_f1_val")
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Q3 — Macro-F1 de validação")
ax.set_xlabel("Neurônios por camada"); ax.set_ylabel("Camadas ocultas")
savefig(fig, "q3_heatmap_val_f1.png")

pivot_time = q3_grid.pivot(index="n_layers", columns="neurons", values="time_s")
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(pivot_time, annot=True, fmt=".0f", cmap="magma", ax=ax)
ax.set_title("Q3 — Tempo de treinamento (s)")
ax.set_xlabel("Neurônios por camada"); ax.set_ylabel("Camadas ocultas")
savefig(fig, "q3_heatmap_time.png")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), q3_top4.iterrows()):
    g = q3_hist[q3_hist["run_id"] == row["run_id"]]
    ax.plot(g["epoch"], g["loss"], label="treino")
    ax.plot(g["epoch"], g["val_loss"], label="validação")
    ax.set_title(row["run_id"])
    ax.set_xlabel("Época"); ax.set_ylabel("Loss")
    ax.legend()
fig.suptitle("Q3 — Curvas das 4 melhores topologias", y=1.02)
savefig(fig, "q3_top4_loss_curves.png")

**Análise Q3.** O aumento de profundidade/largura tende a reduzir a loss de treino, mas pode aumentar o gap treino–validação. As quatro topologias selecionadas equilibram macro-F1 de validação, loss e tempo, e seguem para a Q6.

## 8. Q4 — Quantidade de dados

Frações de 20%, 40%, 60%, 80% e 100% do treino de 56%, com amostragem estratificada. Scaler ajustado no subconjunto; validação fixa; teste não usado.

In [ ]:
Q4_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
q3_best = q3_top4.iloc[0]
Q4_HIDDEN = [int(q3_best["neurons"])] * int(q3_best["n_layers"])
q4_path = RESULTS_CSV / "q4_fractions.csv"
q4_hist_path = RESULTS_CSV / "q4_history.csv"

if q4_path.exists() and q4_hist_path.exists() and not FORCE_RERUN["q4"]:
    q4 = pd.read_csv(q4_path)
    q4_hist = pd.read_csv(q4_hist_path)
    print("Q4 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    for frac in Q4_FRACTIONS:
        run_id = f"q4_frac{int(frac*100)}"
        if frac < 1.0:
            X_sub, y_sub = train_test_split(
                X_train, y_train, train_size=frac, stratify=y_int_train, random_state=SEED)
        else:
            X_sub, y_sub = X_train, y_train
        scaler_sub = StandardScaler()
        X_sub_s = scaler_sub.fit_transform(X_sub).astype("float32")
        X_val_s = scaler_sub.transform(X_val).astype("float32")
        clear_session()
        model = build_mlp(X_sub_s.shape[1], Q4_HIDDEN, q2_lr, q2_mom, seed=SEED)
        cb = ConvergenceCallback()
        train_ds = make_dataset(X_sub_s, y_sub, training=True)
        t0 = time.perf_counter()
        hist = model.fit(train_ds, validation_data=(X_val_s, y_val),
                         epochs=MAX_EPOCHS, callbacks=[cb], verbose=0)
        elapsed = time.perf_counter() - t0
        n_epochs = len(hist.history["loss"])
        status = "diverged" if cb.diverged or not np.isfinite(hist.history["loss"][-1]) else "ok"
        if status == "ok":
            tr_metrics = evaluate_model(model, X_sub_s, y_sub)
            va_metrics = evaluate_model(model, X_val_s, y_val)
        else:
            tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
            va_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
        run_rows.append({
            "run_id": run_id, "fraction": frac, "n_train": int(len(X_sub)),
            "status": status, "converged": bool(cb.converged), "epochs": n_epochs,
            "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
            "loss_val": va_metrics["loss"], "acc_val": va_metrics["accuracy"],
            "macro_f1_val": va_metrics["macro_f1"], "weighted_f1_val": va_metrics["weighted_f1"],
            "time_s": elapsed,
        })
        hist_rows.append(history_to_df(hist, run_id=run_id, fraction=frac))
        print(f"{run_id}: n={len(X_sub)} status={status} val_f1={va_metrics['macro_f1']:.4f}")
    q4 = pd.DataFrame(run_rows)
    q4_hist = pd.concat(hist_rows, ignore_index=True)
    q4.to_csv(q4_path, index=False)
    q4_hist.to_csv(q4_hist_path, index=False)

best = q4.sort_values("macro_f1_val", ascending=False).iloc[0]
full = q4[q4["fraction"] == 1.0].iloc[0]
q4_winner_fraction = 1.0 if full["macro_f1_val"] >= best["macro_f1_val"] - 0.005 else float(best["fraction"])
q4["selected"] = q4["fraction"].eq(q4_winner_fraction)
q4.to_csv(q4_path, index=False)
print("Fração vencedora da Q4:", q4_winner_fraction)
display(q4.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["loss_val", "acc_val", "macro_f1_val"],
                          ["Loss de validação", "Acurácia de validação", "Macro-F1 de validação"]):
    ax.plot(q4["fraction"], q4[col], marker="o", color=PALETTE[0])
    ax.set_title(title); ax.set_xlabel("Fração do treino"); ax.set_ylabel(col)
savefig(fig, "q4_generalization_curves.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q4["fraction"], q4["time_s"], marker="o", color=PALETTE[1])
ax.set_title("Q4 — Tempo de treinamento"); ax.set_xlabel("Fração do treino"); ax.set_ylabel("Segundos")
savefig(fig, "q4_time_vs_fraction.png")

fig, ax = plt.subplots(figsize=(8, 5))
for run_id, g in q4_hist.groupby("run_id"):
    ax.plot(g["epoch"], g["val_loss"], label=run_id, alpha=0.8)
ax.set_title("Q4 — Loss de validação por fração"); ax.set_xlabel("Época"); ax.set_ylabel("Loss")
ax.legend(fontsize=8)
savefig(fig, "q4_loss_curves.png")

**Análise Q4.** As curvas de generalização mostram se o aumento de dados melhora o desempenho ou satura. O tempo cresce com o número de exemplos. A Q4 é uma análise de sensibilidade; o modelo final da Q6 usa todo o desenvolvimento (70%) para maximizar a informação disponível.

## 9. Q5 — Atributos

Análise de informação mútua, correlação e variância somente no treino. Subconjuntos: `all_16`, `low_redundancy`, `top_10_mi`, `compact`, `no_noise`.

In [ ]:
n_features = len(FEATURE_COLS)
mi_train = mutual_info_classif(X_train, y_int_train, random_state=SEED)
var_train = np.var(X_train, axis=0)

rule_features = {}
for rule in ["all_16", "low_redundancy", "top_10_mi", "compact", "no_noise"]:
    rule_features[rule] = select_by_rule(rule, X_train, y_int_train, FEATURE_COLS)

feature_rows = []
for i, f in enumerate(FEATURE_COLS):
    feature_rows.append({
        "feature": f, "variance": float(var_train[i]), "mutual_info": float(mi_train[i]),
        **{rule: bool(f in feats) for rule, feats in rule_features.items()},
    })
q5_features = pd.DataFrame(feature_rows).sort_values("mutual_info", ascending=False)
q5_features.to_csv(RESULTS_CSV / "q5_features.csv", index=False)
display(q5_features.round(5))

q5_path = RESULTS_CSV / "q5_subsets.csv"
q5_hist_path = RESULTS_CSV / "q5_history.csv"

if q5_path.exists() and q5_hist_path.exists() and not FORCE_RERUN["q5"]:
    q5 = pd.read_csv(q5_path)
    q5_hist = pd.read_csv(q5_hist_path)
    print("Q5 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    all_metrics = None
    for rule in ["all_16", "low_redundancy", "top_10_mi", "compact", "no_noise"]:
        feats = rule_features[rule]
        idx = feature_indices(feats, FEATURE_COLS)
        run_id = f"q5_{rule}"
        reused = False
        if rule == "no_noise" and set(feats) == set(FEATURE_COLS):
            reused = True
            tr_metrics = all_metrics["train"]
            va_metrics = all_metrics["val"]
            status = "reused"
            n_epochs = 0
            elapsed = 0.0
            cb = type("CB", (), {"converged": False, "diverged": False})()
            print(f"{run_id}: sem remoções; reutilizando all_16.")
        else:
            clear_session()
            model = build_mlp(len(idx), Q4_HIDDEN, q2_lr, q2_mom, seed=SEED)
            cb = ConvergenceCallback()
            train_ds = make_dataset(X_train_scaled[:, idx], y_train, training=True)
            t0 = time.perf_counter()
            hist = model.fit(train_ds, validation_data=(X_val_scaled[:, idx], y_val),
                             epochs=MAX_EPOCHS, callbacks=[cb], verbose=0)
            elapsed = time.perf_counter() - t0
            n_epochs = len(hist.history["loss"])
            status = "diverged" if cb.diverged or not np.isfinite(hist.history["loss"][-1]) else "ok"
            if status == "ok":
                tr_metrics = evaluate_model(model, X_train_scaled[:, idx], y_train)
                va_metrics = evaluate_model(model, X_val_scaled[:, idx], y_val)
            else:
                tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
                va_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1"]}
            hist_rows.append(history_to_df(hist, run_id=run_id, subset=rule))
            print(f"{run_id}: n={len(idx)} status={status} val_f1={va_metrics['macro_f1']:.4f}")
        if rule == "all_16":
            all_metrics = {"train": tr_metrics, "val": va_metrics}
        run_rows.append({
            "run_id": run_id, "subset": rule, "n_features": len(feats),
            "features": "|".join(feats), "status": status, "reused": reused,
            "converged": bool(cb.converged), "epochs": n_epochs,
            "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
            "loss_val": va_metrics["loss"], "acc_val": va_metrics["accuracy"],
            "macro_f1_val": va_metrics["macro_f1"], "weighted_f1_val": va_metrics["weighted_f1"],
            "time_s": elapsed,
        })
    q5 = pd.DataFrame(run_rows)
    q5_hist = pd.concat(hist_rows, ignore_index=True) if hist_rows else pd.DataFrame()
    q5.to_csv(q5_path, index=False)
    q5_hist.to_csv(q5_hist_path, index=False)

valid = q5[(q5["status"].isin(["ok", "reused"])) & np.isfinite(q5["macro_f1_val"])].copy()
best_f1 = valid["macro_f1_val"].max()
cands = valid[valid["macro_f1_val"] >= best_f1 - 0.005]
q5_winner = cands.sort_values(["n_features", "loss_val", "time_s"], ascending=[True, True, True]).iloc[0]
q5["selected"] = q5["subset"].eq(q5_winner["subset"])
q5.to_csv(q5_path, index=False)
q5_winner_rule = q5_winner["subset"]
q5_winner_features = q5_winner["features"].split("|")
pd.DataFrame({"feature": q5_winner_features}).to_csv(RESULTS_CSV / "q5_winner_features.csv", index=False)
print("Q5 vencedora:", q5_winner_rule, "| n_features =", q5_winner["n_features"])
display(q5.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=q5_features, x="mutual_info", y="feature", palette="viridis", ax=ax)
ax.set_title("Q5 — Informação mútua por atributo"); ax.set_xlabel("MI"); ax.set_ylabel("")
savefig(fig, "q5_mi_ranking.png")

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=q5, x="subset", y="macro_f1_val", palette="colorblind", ax=ax)
ax.set_title("Q5 — Macro-F1 por subconjunto"); ax.set_xlabel(""); ax.set_ylabel("Macro-F1")
ax.tick_params(axis="x", rotation=20)
savefig(fig, "q5_subset_comparison.png")

if len(q5_hist):
    fig, ax = plt.subplots(figsize=(9, 5))
    for run_id, g in q5_hist.groupby("run_id"):
        ax.plot(g["epoch"], g["val_loss"], label=run_id, alpha=0.8)
    ax.set_title("Q5 — Loss de validação por subconjunto")
    ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.legend(fontsize=8)
    savefig(fig, "q5_loss_curves.png")

**Análise Q5.** A informação mútua ordena a relevância; a correlação identifica redundância; variância baixa combinada com MI baixa sinaliza possível ruído. O subconjunto vencedor equilibra macro-F1 e número de atributos.

## 10. Q6 — Avaliação final no teste

As quatro melhores topologias da Q3 usam os hiperparâmetros da Q2 e a regra de atributos da Q5, reajustada no desenvolvimento de 70%. O teste de 30% é usado apenas aqui. As curvas de teste são registradas por callback apenas para visualização; a parada usa exclusivamente a loss de treino.

In [ ]:
Q6_HIDDEN = {row["run_id"]: [int(row["neurons"])] * int(row["n_layers"]) for _, row in q3_top4.iterrows()}
# Reajusta a regra de atributos no desenvolvimento (70%), sem usar o teste
dev_features = select_by_rule(q5_winner_rule, X_dev, y_int_dev, FEATURE_COLS)
dev_idx = feature_indices(dev_features, FEATURE_COLS)
scaler_q6 = StandardScaler()
X_dev_q6 = scaler_q6.fit_transform(X_dev[:, dev_idx]).astype("float32")
X_test_q6 = scaler_q6.transform(X_test[:, dev_idx]).astype("float32")
print("Atributos usados na Q6:", dev_features)

q6_path = RESULTS_CSV / "q6_test.csv"
q6_hist_path = RESULTS_CSV / "q6_history.csv"
q6_predictions = {}

if q6_path.exists() and q6_hist_path.exists() and not FORCE_RERUN["q6"]:
    q6 = pd.read_csv(q6_path)
    q6_hist = pd.read_csv(q6_hist_path)
    print("Q6 carregada do cache.")
else:
    run_rows, hist_rows = [], []
    for _, cfg in q3_top4.iterrows():
        run_id = f"q6_{cfg['run_id']}"
        clear_session()
        model = build_mlp(len(dev_idx), Q6_HIDDEN[cfg["run_id"]], q2_lr, q2_mom, seed=SEED)
        cb = ConvergenceCallback()
        test_cb = TestMetricsCallback(X_test_q6, y_test)
        train_ds = make_dataset(X_dev_q6, y_dev, training=True)
        t0 = time.perf_counter()
        hist = model.fit(train_ds, epochs=MAX_EPOCHS, callbacks=[cb, test_cb], verbose=0)
        elapsed = time.perf_counter() - t0
        n_epochs = len(hist.history["loss"])
        status = "diverged" if cb.diverged or not np.isfinite(hist.history["loss"][-1]) else "ok"
        if status == "ok":
            tr_metrics = evaluate_model(model, X_dev_q6, y_dev)
            te_metrics = evaluate_model(model, X_test_q6, y_test)
            y_true, y_pred = predict_labels(model, X_test_q6, y_test)
            q6_predictions[run_id] = (y_true, y_pred)
        else:
            tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]}
            te_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]}
        run_rows.append({
            "run_id": run_id, "q3_run_id": cfg["run_id"],
            "n_layers": int(cfg["n_layers"]), "neurons": int(cfg["neurons"]),
            "n_params": int(model.count_params()), "status": status,
            "converged": bool(cb.converged), "epochs": n_epochs,
            "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
            "macro_f1_train": tr_metrics["macro_f1"],
            "loss_test": te_metrics["loss"], "acc_test": te_metrics["accuracy"],
            "macro_f1_test": te_metrics["macro_f1"], "weighted_f1_test": te_metrics["weighted_f1"],
            "precision_macro_test": te_metrics["precision_macro"],
            "recall_macro_test": te_metrics["recall_macro"],
            "time_s": elapsed,
        })
        h = history_to_df(hist, run_id=run_id, q3_run_id=cfg["run_id"])
        n = len(h)
        tl = list(test_cb.test_loss)[:n] + [np.nan] * max(0, n - len(test_cb.test_loss))
        ta = list(test_cb.test_accuracy)[:n] + [np.nan] * max(0, n - len(test_cb.test_accuracy))
        h["test_loss"] = tl
        h["test_accuracy"] = ta
        hist_rows.append(h)
        print(f"{run_id}: status={status} epochs={n_epochs} test_f1={te_metrics['macro_f1']:.4f} tempo={elapsed:.1f}s")
    q6 = pd.DataFrame(run_rows)
    q6_hist = pd.concat(hist_rows, ignore_index=True)
    q6.to_csv(q6_path, index=False)
    q6_hist.to_csv(q6_hist_path, index=False)

q6_winner = q6.sort_values(
    ["macro_f1_test", "weighted_f1_test", "loss_test", "time_s"],
    ascending=[False, False, True, True]).iloc[0]
q6["selected"] = q6["run_id"].eq(q6_winner["run_id"])
q6.to_csv(q6_path, index=False)
print("Q6 vencedora:", q6_winner["run_id"], "| macro-F1 teste =", q6_winner["macro_f1_test"])
display(q6.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=q6, x="run_id", y="macro_f1_test", palette="colorblind", ax=ax)
ax.set_title("Q6 — Macro-F1 no teste"); ax.set_xlabel(""); ax.set_ylabel("Macro-F1")
ax.tick_params(axis="x", rotation=20)
savefig(fig, "q6_test_comparison.png")

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for _, row in q6.iterrows():
    g = q6_hist[q6_hist["run_id"] == row["run_id"]]
    axes[0, 0].plot(g["epoch"], g["loss"], label=row["run_id"], alpha=0.8)
    axes[0, 1].plot(g["epoch"], g["test_loss"], label=row["run_id"], alpha=0.8)
    axes[1, 0].plot(g["epoch"], g["accuracy"], label=row["run_id"], alpha=0.8)
    axes[1, 1].plot(g["epoch"], g["test_accuracy"], label=row["run_id"], alpha=0.8)
axes[0, 0].set_title("Loss de treino"); axes[0, 0].set_ylabel("Loss")
axes[0, 1].set_title("Loss de teste"); axes[0, 1].set_ylabel("Loss")
axes[1, 0].set_title("Acurácia de treino"); axes[1, 0].set_ylabel("Acurácia")
axes[1, 1].set_title("Acurácia de teste"); axes[1, 1].set_ylabel("Acurácia")
for ax in axes.ravel():
    ax.set_xlabel("Época"); ax.legend(fontsize=7)
fig.suptitle("Q6 — Curvas de treino e teste", y=1.02)
savefig(fig, "q6_train_test_curves.png")

if q6_winner["run_id"] in q6_predictions:
    y_true, y_pred = q6_predictions[q6_winner["run_id"]]
else:
    # Se carregado do cache, não há modelo; a matriz fica pendente de reexecução.
    y_true, y_pred = None, None
if y_true is not None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title("Q6 — Matriz de confusão da vencedora")
    ax.set_xlabel("Predito"); ax.set_ylabel("Real")
    savefig(fig, "q6_confusion_matrix.png")
    pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names,
                                       output_dict=True, zero_division=0)).T.to_csv(
        RESULTS_CSV / "q6_classification_report.csv")

**Análise Q6.** A comparação no teste mostra qual topologia generaliza melhor sob as mesmas condições de hiperparâmetros e atributos. A vencedora equilibra desempenho e estabilidade e segue para a Q7.

## 11. Q7 — Validação cruzada

k=5 estratificado no desenvolvimento (70%); scaler e regra de atributos reajustados dentro de cada fold. O teste de 30% não é usado.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
q7_path = RESULTS_CSV / "q7_cv.csv"
q7_hist_path = RESULTS_CSV / "q7_history.csv"
q7_folds_path = RESULTS_CSV / "q7_folds.csv"
q7_summary_path = RESULTS_CSV / "q7_summary.csv"

if q7_path.exists() and q7_hist_path.exists() and q7_summary_path.exists() and not FORCE_RERUN["q7"]:
    q7 = pd.read_csv(q7_path)
    q7_hist = pd.read_csv(q7_hist_path)
    q7_summary = pd.read_csv(q7_summary_path)
    print("Q7 carregada do cache.")
else:
    fold_df = pd.DataFrame({
        "row_index": idx_dev,
        "fold": -1,
        "class": label_encoder.inverse_transform(y_int_dev),
    })
    run_rows, hist_rows = [], []
    for fold, (tr_pos, va_pos) in enumerate(skf.split(X_dev, y_int_dev)):
        fold_df.loc[fold_df.index[va_pos], "fold"] = fold
        X_tr_raw, y_tr = X_dev[tr_pos], y_dev[tr_pos]
        X_va_raw, y_va = X_dev[va_pos], y_dev[va_pos]
        feats = select_by_rule(q5_winner_rule, X_tr_raw, np.argmax(y_tr, axis=1), FEATURE_COLS)
        idx = feature_indices(feats, FEATURE_COLS)
        scaler_fold = StandardScaler()
        X_tr = scaler_fold.fit_transform(X_tr_raw[:, idx]).astype("float32")
        X_va = scaler_fold.transform(X_va_raw[:, idx]).astype("float32")
        clear_session()
        model = build_mlp(len(idx), Q6_HIDDEN[q6_winner["q3_run_id"]], q2_lr, q2_mom, seed=SEED)
        cb = ConvergenceCallback()
        train_ds = make_dataset(X_tr, y_tr, training=True)
        t0 = time.perf_counter()
        hist = model.fit(train_ds, validation_data=(X_va, y_va),
                         epochs=MAX_EPOCHS, callbacks=[cb], verbose=0)
        elapsed = time.perf_counter() - t0
        n_epochs = len(hist.history["loss"])
        status = "diverged" if cb.diverged or not np.isfinite(hist.history["loss"][-1]) else "ok"
        if status == "ok":
            tr_metrics = evaluate_model(model, X_tr, y_tr)
            va_metrics = evaluate_model(model, X_va, y_va)
        else:
            tr_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]}
            va_metrics = {k: np.nan for k in ["loss", "accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]}
        run_rows.append({
            "fold": fold, "status": status, "converged": bool(cb.converged),
            "epochs": n_epochs, "n_train": len(tr_pos), "n_val": len(va_pos),
            "loss_train": tr_metrics["loss"], "acc_train": tr_metrics["accuracy"],
            "loss_val": va_metrics["loss"], "acc_val": va_metrics["accuracy"],
            "macro_f1_val": va_metrics["macro_f1"], "weighted_f1_val": va_metrics["weighted_f1"],
            "precision_macro_val": va_metrics["precision_macro"],
            "recall_macro_val": va_metrics["recall_macro"], "time_s": elapsed,
            "features": "|".join(feats),
        })
        hist_rows.append(history_to_df(hist, fold=fold))
        print(f"fold {fold}: status={status} val_f1={va_metrics['macro_f1']:.4f} tempo={elapsed:.1f}s")
    q7 = pd.DataFrame(run_rows)
    q7_hist = pd.concat(hist_rows, ignore_index=True)
    fold_df.to_csv(q7_folds_path, index=False)
    q7.to_csv(q7_path, index=False)
    q7_hist.to_csv(q7_hist_path, index=False)

    summary_rows = []
    for col in ["loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
                "weighted_f1_val", "precision_macro_val", "recall_macro_val", "epochs", "time_s"]:
        vals = q7[col].dropna().astype(float).values
        summary_rows.append({
            "metric": col, "mean": float(np.mean(vals)) if len(vals) else np.nan,
            "variance": float(np.var(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "min": float(np.min(vals)) if len(vals) else np.nan,
            "max": float(np.max(vals)) if len(vals) else np.nan,
        })
    q7_summary = pd.DataFrame(summary_rows)
    q7_summary.to_csv(q7_summary_path, index=False)

display(q7.round(4))
display(q7_summary.round(5))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for fold, g in q7_hist.groupby("fold"):
    axes[0].plot(g["epoch"], g["val_loss"], label=f"fold {fold}", alpha=0.8)
    axes[1].plot(g["epoch"], g["val_accuracy"], label=f"fold {fold}", alpha=0.8)
axes[0].set_title("Q7 — Loss de validação por fold"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss")
axes[1].set_title("Q7 — Acurácia de validação por fold"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("Acurácia")
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7)
savefig(fig, "q7_fold_curves.png")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["loss_val", "acc_val", "macro_f1_val"],
                          ["Loss de validação", "Acurácia de validação", "Macro-F1 de validação"]):
    sns.boxplot(y=q7[col], ax=ax, color=PALETTE[0])
    sns.stripplot(y=q7[col], ax=ax, color="black", size=5)
    ax.set_title(title); ax.set_ylabel(col)
savefig(fig, "q7_metric_boxplots.png")

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=q7_summary[q7_summary["metric"].isin(["loss_val", "acc_val", "macro_f1_val"])],
            x="metric", y="mean", palette="colorblind", ax=ax)
ax.set_title("Q7 — Média entre folds"); ax.set_xlabel(""); ax.set_ylabel("Média")
savefig(fig, "q7_summary.png")

**Análise Q7.** A média e a variância entre folds indicam a robustez. A Q7 usa o desenvolvimento de 70%; o teste de 30% permanece reservado à Q6. Como a Q6 já selecionou a configuração no teste, a Q7 é uma análise de robustez pós-seleção, não um segundo holdout independente.

## 12. Consolidado dos vencedores

In [ ]:
winners = []
# Q2
winners.append({
    "stage": "Q2", "run_id": q2_winner["run_id"],
    "configuration": f"lr={q2_lr}; momentum={q2_mom}",
    "selection_metric": "macro_f1_val", "value": float(q2_winner["macro_f1_val"]),
    "notes": f"converged={bool(q2_winner['converged'])}; epochs={int(q2_winner['epochs'])}",
})
# Q3 top4
for _, row in q3_top4.iterrows():
    winners.append({
        "stage": f"Q3_top{int(row['rank'])}", "run_id": row["run_id"],
        "configuration": f"layers={int(row['n_layers'])}; neurons={int(row['neurons'])}",
        "selection_metric": "macro_f1_val", "value": float(row["macro_f1_val"]),
        "notes": f"loss_val={row['loss_val']:.4f}; time_s={row['time_s']:.1f}",
    })
# Q4
winners.append({
    "stage": "Q4", "run_id": q4[q4["selected"] == True].iloc[0]["run_id"],
    "configuration": f"fraction={q4_winner_fraction}",
    "selection_metric": "macro_f1_val",
    "value": float(q4[q4["selected"] == True].iloc[0]["macro_f1_val"]),
    "notes": "análise de sensibilidade; Q6 usa o desenvolvimento completo",
})
# Q5
winners.append({
    "stage": "Q5", "run_id": q5_winner["run_id"],
    "configuration": f"subset={q5_winner_rule}; n_features={int(q5_winner['n_features'])}",
    "selection_metric": "macro_f1_val", "value": float(q5_winner["macro_f1_val"]),
    "notes": "; ".join(q5_winner_features),
})
# Q6
winners.append({
    "stage": "Q6", "run_id": q6_winner["run_id"],
    "configuration": f"layers={int(q6_winner['n_layers'])}; neurons={int(q6_winner['neurons'])}; "
                     f"lr={q2_lr}; momentum={q2_mom}; features={len(dev_features)}",
    "selection_metric": "macro_f1_test", "value": float(q6_winner["macro_f1_test"]),
    "notes": f"weighted_f1_test={q6_winner['weighted_f1_test']:.4f}",
})
# Q7
q7_macro = q7_summary[q7_summary["metric"] == "macro_f1_val"].iloc[0]
winners.append({
    "stage": "Q7", "run_id": q6_winner["run_id"],
    "configuration": "k=5; desenvolvimento 70%",
    "selection_metric": "macro_f1_val_mean", "value": float(q7_macro["mean"]),
    "notes": f"variance={q7_macro['variance']:.6f}; std={q7_macro['std']:.6f}",
})
winners_df = pd.DataFrame(winners)
winners_df.to_csv(RESULTS_CSV / "winners.csv", index=False)
display(winners_df.round(5))
print("CSVs em", RESULTS_CSV)
print("Figuras em", RESULTS_FIG)

## 13. Encerramento

Todos os resultados foram gravados em `results/csv/` e as figuras em `results/figures/`. O notebook `02_analise_resultados.ipynb` gera as tabelas Markdown para o relatório. O relatório final deve ser preenchido exclusivamente com os valores obtidos.